ResNet for the biggest dataset

In [1]:
import os
import torch
import cv2
import numpy as np
from PIL import Image
from torchvision import models, transforms
from tqdm import tqdm

# --- 1. SETUP (Same as before) ---
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*(list(resnet.children())[:-1]))
resnet.eval()

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_resnet_features(video_path):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0: fps = 30

    frame_features = []

    # Primary Loop: Extract 1 frame per second
    count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if count % int(fps) == 0:
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img)
            input_tensor = preprocess(pil_img).unsqueeze(0)

            with torch.no_grad():
                feature = resnet(input_tensor)
            frame_features.append(feature.flatten().numpy())
        count += 1

    # FALLBACK: If we didn't get any frames, grab the MIDDLE frame
    if not frame_features and total_frames > 0:
        middle_frame_idx = total_frames // 2
        cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame_idx)
        ret, frame = cap.read()
        if ret:
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img)
            input_tensor = preprocess(pil_img).unsqueeze(0)
            with torch.no_grad():
                feature = resnet(input_tensor)
            frame_features.append(feature.flatten().numpy())

    cap.release()

    if not frame_features:
        return np.zeros(2048) # Final safety net

    return np.mean(frame_features, axis=0)

# --- 2. LOAD EXISTING DATA ---
PROJECT_PATH = "Thesis_Data"
existing_features_path = f"{PROJECT_PATH}/visual_features.npy"

if os.path.exists(existing_features_path):
    # Loading allow_pickle=True since it's a dictionary of numpy arrays
    combined_features = np.load(existing_features_path, allow_pickle=True).item()
    print(f"Loaded {len(combined_features)} existing video features.")
else:
    combined_features = {}
    print("No existing features found. Starting fresh.")

# --- 3. PROCESS THE NEW FOLDER ---
# Define the path for the new videos
new_videos_path = f"{PROJECT_PATH}/videos2"
new_video_files = [f for f in os.listdir(new_videos_path) if f.endswith('.mp4')]

print(f"Extracting features from {new_videos_path}...")
for v_file in tqdm(new_video_files):
    v_id = v_file.replace(".mp4", "")
    
    # Skip if we already processed this video ID (optional safety)
    if v_id in combined_features:
        continue
        
    v_path = os.path.join(new_videos_path, v_file)

    try:
        features = extract_resnet_features(v_path)
        combined_features[v_id] = features
    except Exception as e:
        print(f"Error processing {v_id}: {e}")

# --- 4. SAVE COMBINED DATA ---
save_path = f"{PROJECT_PATH}/visual_features_ResNet_combined.npy"
np.save(save_path, combined_features)

print(f"Combined features saved to {save_path}!")
print(f"Total videos processed: {len(combined_features)}")

c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\yasam/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:04<00:00, 21.3MB/s]


Loaded 486 existing video features.
Extracting features from Thesis_Data/videos2...


100%|██████████| 141/141 [36:15<00:00, 15.43s/it]


Combined features saved to Thesis_Data/visual_features_ResNet_combined.npy!
Total videos processed: 627


MFCC for the biggest dataset

In [2]:
import os
import librosa
import numpy as np
from tqdm import tqdm

# --- 1. SETTINGS & PATHS ---
PROJECT_PATH = "Thesis_Data"

def extract_mfcc_features(audio_path):
    # Load audio file (16kHz)
    y, sr = librosa.load(audio_path, sr=16000)

    # Extract MFCCs (n_mfcc=20)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)

    # Aggregate over time: take the mean and standard deviation
    mfccs_mean = np.mean(mfccs, axis=1)
    mfccs_std = np.std(mfccs, axis=1)

    # Combine them into a 40-dimensional vector
    return np.hstack((mfccs_mean, mfccs_std))

# --- 2. LOAD EXISTING AUDIO FEATURES ---
existing_audio_path = f"{PROJECT_PATH}/audio_features.npy"

if os.path.exists(existing_audio_path):
    # Load existing dictionary; allow_pickle=True is required for object arrays (dicts)
    combined_audio_features = np.load(existing_audio_path, allow_pickle=True).item()
    print(f"Loaded {len(combined_audio_features)} existing audio features.")
else:
    combined_audio_features = {}
    print("No existing audio features found. Starting fresh.")

# --- 3. PROCESS THE NEW AUDIO FOLDER (audio2) ---
new_audio_dir = f"{PROJECT_PATH}/audio2"
new_audio_files = [f for f in os.listdir(new_audio_dir) if f.endswith('.wav')]

print(f"Extracting MFCC features from {new_audio_dir}...")
for a_file in tqdm(new_audio_files):
    v_id = a_file.replace(".wav", "")
    
    # Optional: Skip if already in the dictionary to save time
    if v_id in combined_audio_features:
        continue
        
    a_path = os.path.join(new_audio_dir, a_file)

    try:
        features = extract_mfcc_features(a_path)
        combined_audio_features[v_id] = features
    except Exception as e:
        print(f"Error processing {v_id}: {e}")

# --- 4. SAVE THE COMBINED DATASET ---
combined_save_path = f"{PROJECT_PATH}/audio_features_MFCC_combined.npy"
np.save(combined_save_path, combined_audio_features)

print("--- Extraction Complete ---")
print(f"Combined features saved to: {combined_save_path}")
print(f"Total audio profiles in combined file: {len(combined_audio_features)}")

Loaded 486 existing audio features.
Extracting MFCC features from Thesis_Data/audio2...


100%|██████████| 141/141 [01:19<00:00,  1.77it/s]

--- Extraction Complete ---
Combined features saved to: Thesis_Data/audio_features_MFCC_combined.npy
Total audio profiles in combined file: 627


In [4]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# --- 1. LOAD DATA ---
# Load labels
PROJECT_PATH = "Thesis_Data"
LABELS_PATH = "videos_with_sentiment_labels.csv"

df = pd.read_csv(LABELS_PATH)

# Load combined features (ensure you use the _combined files created earlier)
visual_features = np.load(f"{PROJECT_PATH}/visual_features_ResNet_combined.npy", allow_pickle=True).item()
audio_features = np.load(f"{PROJECT_PATH}/audio_features_MFCC_combined.npy", allow_pickle=True).item()

X = []
y = []
valid_vids = []

# --- 2. MULTIMODAL FUSION (Feature Level) ---
print("Aligning features and labels...")
for index, row in df.iterrows():
    v_id = row['video_id']
    label = row['majority_sentiment']
    
    # Check if we have both modalities for this specific video
    if v_id in visual_features and v_id in audio_features:
        vis = visual_features[v_id]   # 2048 dims
        aud = audio_features[v_id]    # 40 dims
        
        # Concatenate features: 2048 + 40 = 2088 dimensions
        combined_vector = np.hstack((vis, aud))
        
        X.append(combined_vector)
        y.append(label)
        valid_vids.append(v_id)

X = np.array(X)
y = np.array(y)

# Encode text labels (e.g., 'positive', 'negative') to integers (0, 1, 2)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Dataset Ready. Total samples: {len(X)} | Feature dimensions: {X.shape[1]}")

# --- 3. TRAIN/TEST SPLIT ---
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# --- 4. RANDOM FOREST (Fast Baseline) ---
print("\n--- Training Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
print(f"RF Accuracy: {accuracy_score(y_test, rf_preds):.4f}")

# --- 5. XGBOOST (High Performance) ---
print("\n--- Training XGBoost ---")
# Use hist tree_method for faster training on large feature sets
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, tree_method='hist')
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
print(f"XGB Accuracy: {accuracy_score(y_test, xgb_preds):.4f}")

# --- 6. SUMMARY REPORT ---
print("\nXGBoost Detailed Report:")
print(classification_report(y_test, xgb_preds, target_names=le.classes_))

Aligning features and labels...
Dataset Ready. Total samples: 446 | Feature dimensions: 2088

--- Training Random Forest ---
RF Accuracy: 0.5333

--- Training XGBoost ---
XGB Accuracy: 0.5000

XGBoost Detailed Report:
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00        15
     Neutral       0.29      0.20      0.24        25
    Positive       0.55      0.80      0.65        50

    accuracy                           0.50        90
   macro avg       0.28      0.33      0.30        90
weighted avg       0.39      0.50      0.43        90



c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

### to handle imbalance

stratify=y_encoded: This ensures that the test set actually contains a proportional number of "Negative" samples. Without this, the test set might randomly end up with almost zero negative samples.

class_weight='balanced' (RF): This penalizes the model more heavily when it misclassifies a "Negative" sample compared to a "Positive" one.

compute_sample_weight (XGB): XGBoost doesn't have a simple "balanced" flag for multi-class tasks, so we manually calculate how much "weight" each row in your training data should have based on its label frequency.

n_estimators & learning_rate: I increased the estimators and lowered the learning rate for XGBoost. Since the dataset is small (~600 samples) but high-dimensional (2088 features), slower learning helps prevent overfitting.

In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# --- 1. LOAD DATA & FEATURES ---
PROJECT_PATH = "Thesis_Data"
LABELS_PATH = "videos_with_sentiment_labels.csv"

df = pd.read_csv(LABELS_PATH)
visual_features = np.load(f"{PROJECT_PATH}/visual_features_ResNet_combined.npy", allow_pickle=True).item()
audio_features = np.load(f"{PROJECT_PATH}/audio_features_MFCC_combined.npy", allow_pickle=True).item()

X = []
y = []

print("Fusing modalities and aligning with labels...")
for _, row in df.iterrows():
    v_id = row['video_id']
    label = row['majority_sentiment']
    
    if v_id in visual_features and v_id in audio_features:
        # Fusion: Concatenating ResNet (2048) and MFCC (40)
        feat_vec = np.hstack((visual_features[v_id], audio_features[v_id]))
        X.append(feat_vec)
        y.append(label)

X = np.array(X)
y = np.array(y)

# Encode Labels (Negative=0, Neutral=1, Positive=2)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# --- 2. TRAIN/TEST SPLIT ---
# Stratify=y_encoded is CRITICAL here to ensure the 69 Negative samples 
# are split fairly between train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# --- 3. RANDOM FOREST (Balanced) ---
print("\n--- Training Random Forest (with Class Weights) ---")
# 'balanced' mode uses the values of y to automatically adjust weights
rf_model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=12, 
    class_weight='balanced', 
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

# --- 4. XGBOOST (Weighted) ---
print("--- Training XGBoost (with Sample Weights) ---")
# For multi-class XGBoost, we calculate sample weights for the training set
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    tree_method='hist',
    random_state=42
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
xgb_preds = xgb_model.predict(X_test)

# --- 5. DYNAMIC RESULTS & EVALUATION ---
def print_results(name, y_true, y_pred):
    print(f"\n[{name} RESULTS]")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=le.classes_))
    # Confusion matrix helps see if 'Negative' is still being ignored
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

print_results("Random Forest", y_test, rf_preds)
print_results("XGBoost", y_test, xgb_preds)

# --- 6. FEATURE IMPORTANCE (Optional Finetuning Insight) ---
importances = rf_model.feature_importances_
vis_imp = np.sum(importances[:2048])
aud_imp = np.sum(importances[2048:])
print(f"\nModality Importance -> Visual: {vis_imp:.2%}, Audio: {aud_imp:.2%}")

Fusing modalities and aligning with labels...

--- Training Random Forest (with Class Weights) ---
--- Training XGBoost (with Sample Weights) ---

[Random Forest RESULTS]
Accuracy: 0.5778
Classification Report:
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00        11
     Neutral       0.40      0.07      0.12        27
    Positive       0.59      0.96      0.73        52

    accuracy                           0.58        90
   macro avg       0.33      0.35      0.28        90
weighted avg       0.46      0.58      0.46        90

Confusion Matrix:
[[ 0  1 10]
 [ 0  2 25]
 [ 0  2 50]]

[XGBoost RESULTS]
Accuracy: 0.5333
Classification Report:
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00        11
     Neutral       0.33      0.26      0.29        27
    Positive       0.61      0.79      0.69        52

    accuracy                           0.53        90
   macro avg       0.3

c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

In [8]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# --- 1. LOAD DATA & FEATURES ---
visual_features_dict = np.load(f"{PROJECT_PATH}/visual_features_ResNet_combined.npy", allow_pickle=True).item()
audio_features_dict = np.load(f"{PROJECT_PATH}/audio_features_MFCC_combined.npy", allow_pickle=True).item()

X_vis = []
X_aud = []
y = []

print("Preparing modalities for Late Fusion...")
for _, row in df.iterrows():
    v_id = row['video_id']
    label = row['majority_sentiment']
    
    if v_id in visual_features_dict and v_id in audio_features_dict:
        X_vis.append(visual_features_dict[v_id])
        X_aud.append(audio_features_dict[v_id])
        y.append(label)

X_vis = np.array(X_vis)
X_aud = np.array(X_aud)
y = np.array(y)

# Encode Labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# --- 2. TRAIN/TEST SPLIT ---
# Using the same random_state and stratify ensures indices match for both modalities
indices = np.arange(len(y_encoded))
X_vis_train, X_vis_test, X_aud_train, X_aud_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_vis, X_aud, y_encoded, indices, test_size=0.2, random_state=42, stratify=y_encoded
)

# --- 3. MODEL 1: VISUAL RANDOM FOREST ---
print("Training Visual RF...")
rf_vis = RandomForestClassifier(
    n_estimators=200, 
    max_depth=12, 
    class_weight='balanced', 
    random_state=42
)
rf_vis.fit(X_vis_train, y_train)

# --- 4. MODEL 2: AUDIO RANDOM FOREST ---
print("Training Audio RF...")
rf_aud = RandomForestClassifier(
    n_estimators=200, 
    max_depth=12, 
    class_weight='balanced', 
    random_state=42
)
rf_aud.fit(X_aud_train, y_train)

# --- 5. LATE FUSION (Soft Voting / Probability Averaging) ---
print("Applying Late Fusion...")

# Get probability distributions for each class
# shape: (n_samples, n_classes)
probs_vis = rf_vis.predict_proba(X_vis_test)
probs_aud = rf_aud.predict_proba(X_aud_test)

# Average the probabilities
# You can also use weighted average if one modality is much stronger, e.g., (0.7*vis + 0.3*aud)
combined_probs = (probs_vis + probs_aud) / 2

# Final prediction is the class with the highest average probability
final_preds = np.argmax(combined_probs, axis=1)

# --- 6. DYNAMIC EVALUATION ---
print("\n" + "="*30)
print("LATE FUSION: RF(Visual) + RF(Audio)")
print("="*30)
print(f"Accuracy: {accuracy_score(y_test, final_preds):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, final_preds, target_names=le.classes_))

print("Confusion Matrix:")
# Rows = Truth, Columns = Predicted
print(confusion_matrix(y_test, final_preds))

# --- 7. INDIVIDUAL MODEL CHECK (For Finetuning) ---
vis_only_acc = accuracy_score(y_test, np.argmax(probs_vis, axis=1))
aud_only_acc = accuracy_score(y_test, np.argmax(probs_aud, axis=1))
print(f"\nBaseline - Visual Only Acc: {vis_only_acc:.4f}")
print(f"Baseline - Audio Only Acc:  {aud_only_acc:.4f}")

Preparing modalities for Late Fusion...
Training Visual RF...
Training Audio RF...
Applying Late Fusion...

LATE FUSION: RF(Visual) + RF(Audio)
Accuracy: 0.5778

Classification Report:
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00        11
     Neutral       0.33      0.07      0.12        27
    Positive       0.60      0.96      0.74        52

    accuracy                           0.58        90
   macro avg       0.31      0.35      0.29        90
weighted avg       0.45      0.58      0.46        90

Confusion Matrix:
[[ 0  2  9]
 [ 1  2 24]
 [ 0  2 50]]

Baseline - Visual Only Acc: 0.5667
Baseline - Audio Only Acc:  0.5667
